# Megatron-Style Tensor Parallelism (Simulated Ranks) — Solution

**Difficulty**: 🔴 Hard

### Problem Statement

The weights of a 70B-parameter model take 140 GB in bf16 — more than one 80 GB GPU holds — and
even a model that fits may run its matmuls too slowly for the latency you need. **Tensor
parallelism** (TP) splits every large matmul of a transformer layer across $W$ GPUs. The
standard recipe comes from Megatron-LM (Shoeybi et al., 2019, §3): inside each block, split the
first linear by its **output** features ("column-parallel") and the second by its **input**
features ("row-parallel"). The activation between them stays local, and the whole block needs a
single all-reduce in forward and a single one in backward:

```
            column-parallel (no comm)             row-parallel             all-reduce
 X ──f──►  H_r = act(X W1_rᵀ + b1_r)  ──►  P_r = H_r W2_rᵀ  ──g──►  Z = Σ_r P_r + b2
 (every     shape (B, T, ffn/W) on rank r      (B, T, d) partial sum      (B, T, d) on every rank
  rank)
```

Self-attention is split the same way: each rank owns $H/W$ whole heads (its rows of
$W_q, W_k, W_v$ are column-parallel), runs attention for them locally, and the output projection
$W_o$ is row-parallel — again one all-reduce. That is two all-reduces in forward and two in
backward for every transformer layer (the paper's Fig. 4): frequent, blocking communication,
which is why TP is normally kept inside one node, on NVLink.

The two communication points are the paper's **conjugate operators**:

- $f$ (entering the block): identity in forward, **all-reduce** in backward;
- $g$ (leaving the block): **all-reduce** in forward, identity in backward.

**Notation trap.** Megatron writes $Y = XA$. PyTorch's `nn.Linear` computes $y = xW^\top + b$
with `weight` of shape `(out_features, in_features)`, so $A = W^\top$:

| Megatron | `nn.Linear` weight `(out, in)` | bias |
|---|---|---|
| column-parallel: $A = [A_1, \dots, A_W]$ split by **columns** | split along **dim 0** (output features): rank $r$ holds rows $[r\frac{out}{W}, (r+1)\frac{out}{W})$ | split the same way |
| row-parallel: $A$ split by **rows**, input $X = [X_1, \dots, X_W]$ | split along **dim 1** (input features): rank $r$ holds columns $[r\frac{in}{W}, (r+1)\frac{in}{W})$ | full size, replicated, added **once**, after the all-reduce |

Megatron-LM stores its shards in exactly this layout: `ColumnParallelLinear.weight` is
`(output_size_per_partition, input_size)` and `RowParallelLinear.weight` is
`(output_size, input_size_per_partition)` (`megatron/core/tensor_parallel/layers.py`).

**The simulation.** You will run $W$ ranks in one process; a rank is a list index. `xs[r]` is
rank $r$'s copy of the block input and `shards[r]` is a dict of rank $r$'s weight shards. The
only way ranks talk is `comm.all_reduce(list_of_W_tensors)`, which returns $W$ new tensors that
each hold the elementwise **sum**. Like NCCL, it is invisible to autograd — which is exactly why
Megatron wraps it in $f$ and $g$ (`_CopyToModelParallelRegion` and
`_ReduceFromModelParallelRegion` in `megatron/core/tensor_parallel/mappings.py`), and so will you.

**Simulating backward.** On a real cluster every rank computes the same loss from its own copy
of $Z$ and calls `backward()` on it. Here we add up the $W$ per-rank losses and call
`backward()` once. For the sharded gradients to come out right, each rank's partial $P_r$ must
receive exactly $\partial L/\partial Z$ — that is $g$'s identity backward. Ordinary autograd
through a sum would hand every partial the gradient of all $W$ outputs, $W\,\partial L/\partial Z$.

---

### Requirements

1. **`shard_column(weight, bias, world_size) -> list[(w_r, b_r)]`** — column-parallel sharding of
   an `nn.Linear`. `weight` is `(out, in)`, `bias` is `(out,)` or `None`. Return $W$ pairs:
   `w_r` = rows $[r\frac{out}{W}, (r+1)\frac{out}{W})$ of `weight` (shape `(out // W, in)`),
   `b_r` = the same slice of `bias` (`None` when `bias` is `None`). Shards are **copies**
   (each rank owns its memory), never views. `out % W != 0` → `ValueError`.
2. **`shard_row(weight, world_size) -> list[w_r]`** — row-parallel sharding: `w_r` = columns
   $[r\frac{in}{W}, (r+1)\frac{in}{W})$ of `weight`, shape `(out, in // W)`, copies.
   `in % W != 0` → `ValueError`. (The bias of a row-parallel layer is not sharded.)
3. **`shard_qkv(w_qkv, b_qkv, n_heads, world_size) -> list[(w_r, b_r)]`** — the dense fused
   projection is `w_qkv = [W_q; W_k; W_v]` of shape `(3d, d)` (the layout of
   `nn.MultiheadAttention.in_proj_weight`), with head $h$ of $Q$ in rows
   $[h\,d_h, (h+1)\,d_h)$ of $W_q$, $d_h = d / H$ (same for $K$, $V$). Rank $r$ owns the
   contiguous heads $[r\frac{H}{W}, (r+1)\frac{H}{W})$ and gets
   `w_r = cat([its rows of W_q, its rows of W_k, its rows of W_v])`, shape `(3d // W, d)`, and
   `b_r` likewise, shape `(3d // W,)`. `n_heads % W != 0` → `ValueError` (even when `d % W == 0`:
   that would cut a head in half).
4. **`copy_to_tp(xs, comm) -> list`** — Megatron's $f$ over the $W$ ranks, implemented with a
   `torch.autograd.Function`. Forward returns the $W$ inputs unchanged and makes **no**
   collective call; backward makes **one** `comm.all_reduce` of the $W$ incoming gradients, so
   every `xs[r]` receives the sum.
5. **`reduce_from_tp(partials, comm) -> list`** — Megatron's $g$: forward makes **one**
   `comm.all_reduce` (sum, not mean) and returns the $W$ results; backward is the identity (rank
   $r$'s partial receives rank $r$'s output gradient) with no communication.
6. **`tp_mlp_forward(xs, shards, act, comm) -> list[Tensor]`** — `xs[r]`: `(B, T, d)`;
   `shards[r] = {"w1": (ffn/W, d), "b1": (ffn/W,), "w2": (d, ffn/W), "b2": (d,)}` (`w1`/`b1`
   column-parallel, `w2` row-parallel, `b2` replicated); `act` is an elementwise function such as
   `F.gelu`. Implement the paper's Fig. 3a with your $f$ and $g$ (the diagram above). Return the
   $W$ per-rank outputs, each `(B, T, d)` and equal to the dense MLP
   `act(x @ W1.T + b1) @ W2.T + b2`.
7. **`tp_attention_forward(xs, shards, n_heads, comm) -> list[Tensor]`** —
   `shards[r] = {"w_qkv": (3d/W, d), "b_qkv": (3d/W,), "w_o": (d, d/W), "b_o": (d,)}` with the
   per-rank `[Q_r; K_r; V_r]` layout of `shard_qkv`. The dense reference is causal multi-head
   self-attention: `q, k, v = F.linear(x, w_qkv, b_qkv).chunk(3, -1)`, split each into
   `n_heads` heads of size $d_h$, $\mathrm{softmax}(qk^\top/\sqrt{d_h} + \text{causal mask})\,v$,
   merge the heads, then `F.linear(·, w_o, b_o)`. Rank $r$ runs its `n_heads // W` heads, applies
   its row-parallel slice of $W_o$ (no bias), then $g$, then `+ b_o`. `n_heads % W != 0` →
   `ValueError`.
8. **Communication budget**: exactly one all-reduce in forward and one in backward per MLP and
   per attention block. With `world_size == 1` the collective is a no-op: you may call it or skip
   it (Megatron's `_reduce` returns early).
9. **Gradients**: with leaf tensors as inputs and shards, `sum((out_r * G).sum() for out_r in outs).backward()`
   must give every shard the matching slice of the dense model's gradient, every replicated bias
   the dense bias gradient, and every `xs[r]` the full dense $\partial L/\partial X$.

---

### Constraints

- ✅ `torch.autograd.Function` with `forward(ctx, comm, *tensors)` / `backward(ctx, *grads)`
  (return `None` as the gradient of `comm`), `F.linear`,
  `F.scaled_dot_product_attention(..., is_causal=True)`.
- ✅ The grader's communicator records exactly what every rank hands to `all_reduce`, and a
  tampering communicator checks that your output is built from what `all_reduce` returns.
- ❌ No `torch.distributed`.
- ❌ Rank $r$ may only use `xs[r]`, `shards[r]` and what collectives return — never concatenate
  the shards back into full weights, and never read `shards[j]` for $j \ne r$.
- ❌ Don't call `comm.all_reduce` outside your $f$/$g$ Functions (it is not differentiable).

---

<details>
  <summary>💡 Hint</summary>

  - Write the dense MLP as two matmuls. If rank $r$ only has some *rows* of $W_1$, which part of
    $XW_1^\top$ can it compute? If it only has some *columns* of $W_2$, what does
    $H_r W_{2,r}^\top$ represent?
  - Is $\mathrm{GeLU}(a + b) = \mathrm{GeLU}(a) + \mathrm{GeLU}(b)$? Which split lets the
    activation run without talking to other ranks?
  - In backward, rank $r$'s loss replica already gives it $\partial L/\partial Z$. How much of
    $\partial L/\partial X$ can rank $r$ compute on its own?
  - Fused QKV: draw the $3d$ rows of `w_qkv` and colour the ones rank 0 needs.

</details>

In [ ]:
import math

import torch
import torch.nn.functional as F

In [ ]:
class SimComm:
    """W simulated tensor-parallel ranks in one process.

    all_reduce(tensors): tensors[r] is rank r's contribution (all the same shape). Returns a list
    of W NEW tensors, each holding the elementwise SUM. Like NCCL it is invisible to autograd: the
    results never require grad. `calls` counts the collectives, `log` records their shapes.
    """

    def __init__(self, world_size):
        self.world_size = world_size
        self.calls = 0
        self.log = []

    def all_reduce(self, tensors):
        tensors = list(tensors)
        assert len(tensors) == self.world_size, "all_reduce needs exactly one tensor per rank"
        assert all(t.shape == tensors[0].shape for t in tensors), "every rank must send the same shape"
        self.calls += 1
        self.log.append(tuple(tensors[0].shape))
        with torch.no_grad():
            total = torch.stack([t.detach() for t in tensors]).sum(0)
        return [total.clone() for _ in tensors]


def make_mlp_params(d, ffn, seed=0):
    """Dense MLP weights in nn.Linear layout (float64 so the tests can compare tightly)."""
    g = torch.Generator().manual_seed(seed)
    r = lambda *s: torch.randn(*s, generator=g, dtype=torch.float64)
    return {"w1": r(ffn, d) / math.sqrt(d), "b1": r(ffn), "w2": r(d, ffn) / math.sqrt(ffn), "b2": r(d)}


def make_attn_params(d, seed=0):
    """Dense attention weights: fused w_qkv = [W_q; W_k; W_v] (3d, d) and the output projection."""
    g = torch.Generator().manual_seed(seed)
    r = lambda *s: torch.randn(*s, generator=g, dtype=torch.float64)
    return {"w_qkv": r(3 * d, d) / math.sqrt(d), "b_qkv": r(3 * d),
            "w_o": r(d, d) / math.sqrt(d), "b_o": r(d)}


def dense_mlp(x, p, act):
    return F.linear(act(F.linear(x, p["w1"], p["b1"])), p["w2"], p["b2"])


def dense_attention(x, p, n_heads):
    B, T, d = x.shape
    q, k, v = F.linear(x, p["w_qkv"], p["b_qkv"]).chunk(3, dim=-1)
    q, k, v = (t.view(B, T, n_heads, d // n_heads).transpose(1, 2) for t in (q, k, v))
    o = F.scaled_dot_product_attention(q, k, v, is_causal=True)          # (B, H, T, d_h)
    return F.linear(o.transpose(1, 2).reshape(B, T, d), p["w_o"], p["b_o"])


comm = SimComm(2)
print(comm.all_reduce([torch.ones(3), 2 * torch.ones(3)]), "calls:", comm.calls)

In [ ]:
def shard_column(weight, bias, world_size):
    """Column-parallel sharding of an nn.Linear: split the OUTPUT features (dim 0).

    Args:
        weight: (out_features, in_features) full weight.
        bias: (out_features,) full bias, or None.
        world_size: number of ranks W; must divide out_features (else ValueError).

    Returns:
        list of W pairs (w_r, b_r): w_r is (out_features // W, in_features), rows
        [r*out/W, (r+1)*out/W) of weight; b_r is the same slice of bias (None if bias is None).
        Copies, not views.
    """
    out_features = weight.shape[0]
    if out_features % world_size != 0:
        raise ValueError(f"out_features={out_features} is not divisible by world_size={world_size}")
    ws = [w.clone() for w in weight.detach().chunk(world_size, dim=0)]
    bs = ([b.clone() for b in bias.detach().chunk(world_size, dim=0)]
          if bias is not None else [None] * world_size)
    return list(zip(ws, bs))


def shard_row(weight, world_size):
    """Row-parallel sharding of an nn.Linear: split the INPUT features (dim 1).

    Args:
        weight: (out_features, in_features) full weight.
        world_size: number of ranks W; must divide in_features (else ValueError).

    Returns:
        list of W tensors of shape (out_features, in_features // W): columns
        [r*in/W, (r+1)*in/W) of weight. Copies, not views.
    """
    in_features = weight.shape[1]
    if in_features % world_size != 0:
        raise ValueError(f"in_features={in_features} is not divisible by world_size={world_size}")
    return [w.clone() for w in weight.detach().chunk(world_size, dim=1)]


def shard_qkv(w_qkv, b_qkv, n_heads, world_size):
    """Head-aligned sharding of a fused [W_q; W_k; W_v] projection.

    Args:
        w_qkv: (3*d, d) — rows [0, d) are W_q, [d, 2d) are W_k, [2d, 3d) are W_v; head h of each
               uses rows [h*d_h, (h+1)*d_h) of its block, d_h = d // n_heads.
        b_qkv: (3*d,) bias in the same layout.
        n_heads: number of heads H; must be divisible by world_size (else ValueError).
        world_size: number of ranks W.

    Returns:
        list of W pairs (w_r, b_r): w_r (3*d // W, d) = [rank r's rows of W_q; of W_k; of W_v],
        b_r (3*d // W,) likewise. Rank r owns heads [r*H/W, (r+1)*H/W). Copies, not views.
    """
    if n_heads % world_size != 0:
        raise ValueError(f"n_heads={n_heads} is not divisible by world_size={world_size}")
    # Shard W_q, W_k and W_v separately (contiguous heads per rank), then fuse per rank.
    per_block = [shard_column(w, b, world_size)
                 for w, b in zip(w_qkv.chunk(3, dim=0), b_qkv.chunk(3, dim=0))]
    return [(torch.cat([blk[r][0] for blk in per_block]), torch.cat([blk[r][1] for blk in per_block]))
            for r in range(world_size)]

In [ ]:
class CopyToTP(torch.autograd.Function):
    """Megatron's f: identity in forward, all-reduce of the W gradients in backward."""

    @staticmethod
    def forward(ctx, comm, *xs):
        ctx.comm = comm
        return tuple(x.view_as(x) for x in xs)       # identity; view_as gives autograd fresh outputs

    @staticmethod
    def backward(ctx, *grads):
        # Rank r's input fed only rank r's shard, so its gradient is a PARTIAL dL/dX;
        # the true dL/dX (needed on every rank, e.g. for the residual stream) is their sum.
        return (None, *ctx.comm.all_reduce(list(grads)))


class ReduceFromTP(torch.autograd.Function):
    """Megatron's g: all-reduce (SUM) in forward, identity in backward."""

    @staticmethod
    def forward(ctx, comm, *partials):
        return tuple(comm.all_reduce(list(partials)))

    @staticmethod
    def backward(ctx, *grads):
        # Z = sum_r P_r, so dL/dP_r = dL/dZ — which rank r already holds (Z is replicated).
        return (None, *grads)


def copy_to_tp(xs, comm):
    """f over W ranks. xs: list of W tensors (rank r's input). Returns a list of W tensors."""
    return list(CopyToTP.apply(comm, *xs))


def reduce_from_tp(partials, comm):
    """g over W ranks. partials: list of W same-shape tensors. Returns W copies of their sum."""
    return list(ReduceFromTP.apply(comm, *partials))

In [ ]:
def tp_mlp_forward(xs, shards, act, comm):
    """Megatron tensor-parallel MLP: column-parallel W1, elementwise act, row-parallel W2.

    Args:
        xs: list of W tensors (B, T, d) — every rank's copy of the same block input.
        shards: list of W dicts {"w1": (ffn/W, d), "b1": (ffn/W,), "w2": (d, ffn/W), "b2": (d,)}.
        act: elementwise activation, e.g. F.gelu.
        comm: SimComm with comm.world_size == W.

    Returns:
        list of W tensors (B, T, d), each equal to act(x @ W1.T + b1) @ W2.T + b2.
        Exactly one all-reduce in forward (inside g) and one in backward (inside f).
    """
    xs = copy_to_tp(xs, comm)                                       # f
    partials = [act(F.linear(x, s["w1"], s["b1"])) @ s["w2"].T      # local; no b2 here
                for x, s in zip(xs, shards)]
    zs = reduce_from_tp(partials, comm)                             # g: the one all-reduce
    return [z + s["b2"] for z, s in zip(zs, shards)]                # replicated bias, once

In [ ]:
def tp_attention_forward(xs, shards, n_heads, comm):
    """Megatron tensor-parallel causal self-attention: heads split across ranks.

    Args:
        xs: list of W tensors (B, T, d) — every rank's copy of the same block input.
        shards: list of W dicts {"w_qkv": (3d/W, d), "b_qkv": (3d/W,), "w_o": (d, d/W), "b_o": (d,)};
                w_qkv / b_qkv in the per-rank [Q_r; K_r; V_r] layout produced by shard_qkv.
        n_heads: total number of heads H (must be divisible by W, else ValueError).
        comm: SimComm with comm.world_size == W.

    Returns:
        list of W tensors (B, T, d), each equal to dense_attention(x, params, n_heads).
        Exactly one all-reduce in forward and one in backward.
    """
    W = len(shards)
    if n_heads % W != 0:
        raise ValueError(f"n_heads={n_heads} is not divisible by world_size={W}")
    h_local = n_heads // W
    xs = copy_to_tp(xs, comm)                                       # f
    partials = []
    for x, s in zip(xs, shards):
        B, T, _ = x.shape
        q, k, v = F.linear(x, s["w_qkv"], s["b_qkv"]).chunk(3, dim=-1)   # each (B, T, h_local*d_h)
        d_h = q.shape[-1] // h_local
        q, k, v = (t.view(B, T, h_local, d_h).transpose(1, 2) for t in (q, k, v))
        o = F.scaled_dot_product_attention(q, k, v, is_causal=True)      # (B, h_local, T, d_h)
        o = o.transpose(1, 2).reshape(B, T, h_local * d_h)
        partials.append(o @ s["w_o"].T)                             # row-parallel, no bias
    zs = reduce_from_tp(partials, comm)                             # g
    return [z + s["b_o"] for z, s in zip(zs, shards)]

In [ ]:
# Helpers used by the tests: build each rank's shard dict with YOUR sharding functions.
def build_mlp_shards(p, world_size):
    cols = shard_column(p["w1"], p["b1"], world_size)
    rows = shard_row(p["w2"], world_size)
    return [{"w1": w1, "b1": b1, "w2": w2, "b2": p["b2"].clone()} for (w1, b1), w2 in zip(cols, rows)]


def build_attn_shards(p, n_heads, world_size):
    qkv = shard_qkv(p["w_qkv"], p["b_qkv"], n_heads, world_size)
    wo = shard_row(p["w_o"], world_size)
    return [{"w_qkv": w, "b_qkv": b, "w_o": o, "b_o": p["b_o"].clone()} for (w, b), o in zip(qkv, wo)]


def leaves(shards):
    """Turn every shard into its own leaf tensor so it receives a .grad."""
    return [{k: v.detach().clone().requires_grad_() for k, v in s.items()} for s in shards]

In [ ]:
# Test 1: sharding layouts
Wt = torch.arange(24.0, dtype=torch.float64).view(6, 4)       # nn.Linear(4, 6).weight layout
bt = torch.arange(6.0, dtype=torch.float64)
cols = shard_column(Wt, bt, 2)
assert [tuple(w.shape) for w, _ in cols] == [(3, 4), (3, 4)], "column-parallel splits dim 0 (outputs)"
assert torch.equal(cols[1][0], Wt[3:]) and torch.equal(cols[1][1], bt[3:])
assert all(b is None for _, b in shard_column(Wt, None, 3))
rows = shard_row(Wt, 2)
assert [tuple(w.shape) for w in rows] == [(6, 2), (6, 2)], "row-parallel splits dim 1 (inputs)"
assert torch.equal(rows[1], Wt[:, 2:])
cols[0][0].zero_()
assert Wt[0, 1] == 1, "shards must be copies, not views"
for bad in (lambda: shard_column(Wt, bt, 4), lambda: shard_row(Wt, 3)):
    try:
        bad()
        raise AssertionError("expected ValueError for a size that does not divide by world_size")
    except ValueError:
        pass

# Fused QKV, d = 4, two heads of size 2, W = 2: rank 1 owns head 1 = rows 2:4 of each of Q, K, V
w_qkv = torch.arange(48.0, dtype=torch.float64).view(12, 4)
b_qkv = torch.arange(12.0, dtype=torch.float64)
qkv = shard_qkv(w_qkv, b_qkv, n_heads=2, world_size=2)
assert tuple(qkv[1][0].shape) == (6, 4)
assert qkv[1][1].tolist() == [2.0, 3.0, 6.0, 7.0, 10.0, 11.0], qkv[1][1]
assert torch.equal(qkv[1][0], w_qkv[[2, 3, 6, 7, 10, 11]])
try:
    shard_qkv(w_qkv, b_qkv, n_heads=2, world_size=4)          # d % 4 == 0, but 2 heads on 4 ranks
    raise AssertionError("expected ValueError: heads must divide over the ranks")
except ValueError:
    pass
print("Test 1 passed: column / row / head-aligned QKV sharding")

In [ ]:
# Test 2: f and g — forward/backward semantics and communication counts
W = 3
grads = [torch.randn(2, 4, dtype=torch.float64) for _ in range(W)]

comm = SimComm(W)
xs = [torch.randn(2, 4, dtype=torch.float64, requires_grad=True) for _ in range(W)]
ys = copy_to_tp(xs, comm)
assert comm.calls == 0, "f is the identity in forward: no communication"
assert all(torch.equal(y, x) for x, y in zip(xs, ys))
sum((y * g).sum() for y, g in zip(ys, grads)).backward()
assert comm.calls == 1, "f's backward: exactly one all-reduce"
assert all(torch.allclose(x.grad, sum(grads)) for x in xs), "f's backward must SUM the W gradients"

comm = SimComm(W)
ps = [torch.randn(2, 4, dtype=torch.float64, requires_grad=True) for _ in range(W)]
zs = reduce_from_tp(ps, comm)
assert comm.calls == 1, "g's forward: exactly one all-reduce"
assert all(torch.allclose(z, sum(p.detach() for p in ps)) for z in zs), "g must SUM (not average)"
sum((z * g).sum() for z, g in zip(zs, grads)).backward()
assert comm.calls == 1, "g's backward is the identity: no communication"
assert all(torch.allclose(p.grad, g) for p, g in zip(ps, grads))
print("Test 2 passed: f and g are conjugate")

In [ ]:
# Test 3: TP MLP == dense MLP for W in {1, 2, 4}, with a single all-reduce
D, FFN, B, T = 8, 32, 2, 5
p_mlp = make_mlp_params(D, FFN, seed=0)
x = torch.randn(B, T, D, dtype=torch.float64)
for act in (F.gelu, torch.tanh):
    ref = dense_mlp(x, p_mlp, act)
    for W in (1, 2, 4):
        comm = SimComm(W)
        outs = tp_mlp_forward([x.clone() for _ in range(W)], build_mlp_shards(p_mlp, W), act, comm)
        assert len(outs) == W, "one output per rank"
        for r, o in enumerate(outs):
            err = (o - ref).abs().max().item()
            assert torch.allclose(o, ref, atol=1e-12), f"W={W}, rank {r}: TP MLP != dense (max err {err:.2e})"
        assert W == 1 or comm.calls == 1, f"W={W}: expected 1 all-reduce, got {comm.calls}"
        print(f"{act.__name__:>5}  W={W}: matches dense, all-reduces: {comm.log}")
print("Test 3 passed")

In [ ]:
# Test 4: TP attention == dense causal attention, heads split over ranks
N_HEADS = 4
p_attn = make_attn_params(D, seed=1)
ref = dense_attention(x, p_attn, N_HEADS)
for W in (1, 2, 4):
    comm = SimComm(W)
    outs = tp_attention_forward([x.clone() for _ in range(W)], build_attn_shards(p_attn, N_HEADS, W),
                                N_HEADS, comm)
    for r, o in enumerate(outs):
        err = (o - ref).abs().max().item()
        assert torch.allclose(o, ref, atol=1e-12), f"W={W}, rank {r}: TP attention != dense (max err {err:.2e})"
    assert W == 1 or comm.calls == 1, f"W={W}: expected 1 all-reduce, got {comm.calls}"
    print(f"W={W}: matches dense, all-reduces: {comm.log}")
try:
    build_attn_shards(p_attn, N_HEADS, 3)
    raise AssertionError("4 heads on 3 ranks must raise ValueError")
except ValueError:
    pass
print("Test 4 passed")

In [ ]:
# Test 5: backward — every shard's gradient is the matching slice of the dense gradient
G = torch.randn(B, T, D, dtype=torch.float64)

def dense_grads(fn, p, *args):
    pd = {k: v.clone().requires_grad_() for k, v in p.items()}
    xd = x.clone().requires_grad_()
    (fn(xd, pd, *args) * G).sum().backward()
    return {k: v.grad for k, v in pd.items()}, xd.grad

def qkv_slice(g, W, r):          # rank r's [Q_r; K_r; V_r] rows of a fused gradient
    return torch.cat([blk.chunk(W, 0)[r] for blk in g.chunk(3, 0)])

slicers = {"w1": lambda g, W, r: g.chunk(W, 0)[r], "b1": lambda g, W, r: g.chunk(W, 0)[r],
           "w2": lambda g, W, r: g.chunk(W, 1)[r], "b2": lambda g, W, r: g,
           "w_qkv": qkv_slice, "b_qkv": qkv_slice,
           "w_o": lambda g, W, r: g.chunk(W, 1)[r], "b_o": lambda g, W, r: g}

for name, fn, p, extra, build, tp in (
        ("MLP", dense_mlp, p_mlp, F.gelu, lambda W: build_mlp_shards(p_mlp, W), tp_mlp_forward),
        ("attention", dense_attention, p_attn, N_HEADS,
         lambda W: build_attn_shards(p_attn, N_HEADS, W), tp_attention_forward)):
    g_dense, gx_dense = dense_grads(fn, p, extra)
    for W in (2, 4):
        shards = leaves(build(W))
        xs = [x.clone().requires_grad_() for _ in range(W)]
        comm = SimComm(W)
        outs = tp(xs, shards, extra, comm)
        sum((o * G).sum() for o in outs).backward()          # every rank's copy of the same loss
        assert comm.calls == 2, f"{name}, W={W}: expected 1 all-reduce forward + 1 backward, got {comm.calls}"
        for r in range(W):
            for k, v in shards[r].items():
                want = slicers[k](g_dense[k], W, r)
                assert v.grad is not None and torch.allclose(v.grad, want, atol=1e-10), \
                    f"{name}, W={W}: rank {r}'s grad of {k} != dense slice"
            assert torch.allclose(xs[r].grad, gx_dense, atol=1e-10), \
                f"{name}, W={W}: xs[{r}].grad should be the full dense dL/dX"
    print(f"{name}: shard gradients == dense gradient slices for W in (2, 4)")
print("Test 5 passed")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("tensor-parallel", shard_column=shard_column, shard_row=shard_row, shard_qkv=shard_qkv,
      copy_to_tp=copy_to_tp, reduce_from_tp=reduce_from_tp,
      tp_mlp_forward=tp_mlp_forward, tp_attention_forward=tp_attention_forward)

### Why it works

**Column-parallel first linear.** Row $j$ of $W_1$ produces output feature $j$, so
$XW_{1,r}^\top + b_{1,r}$ is exactly feature slice $r$ of $XW_1^\top + b_1$. The activation is
elementwise, so $H_r = \mathrm{act}(XW_{1,r}^\top + b_{1,r})$ is the same slice of $H$ — no
communication. (Splitting $W_1$ by *input* features instead would leave each rank with a partial
pre-activation, and $\mathrm{GeLU}(a+b) \ne \mathrm{GeLU}(a) + \mathrm{GeLU}(b)$ would force an
all-reduce before the activation — the paper's argument in §3.)

**Row-parallel second linear.** A matmul sums over its inner dimension, so splitting that
dimension yields partial sums:
$$HW_2^\top = \sum_{r=0}^{W-1} H_r\,W_{2,r}^\top ,$$
where $W_{2,r}$ holds the columns of $W_2$ for rank $r$'s features. Hence one all-reduce with
**sum**, and $b_2$ added once afterwards (adding it inside every $P_r$ gives $W\,b_2$).

**Attention.** Heads do not interact until the output projection: head $h$ only uses rows
$[h\,d_h, (h+1)\,d_h)$ of $W_q$, $W_k$, $W_v$. Concatenating the heads and multiplying by
$W_o^\top$ is again a sum over the inner (head-feature) dimension,
$OW_o^\top = \sum_r O_r W_{o,r}^\top$ — the same column/row pairing as the MLP.

**Backward: $f$ and $g$ are conjugate.** $Z = \sum_r P_r + b_2$ gives
$\partial L/\partial P_r = \partial L/\partial Z$, which every rank already holds because $Z$ is
replicated — so $g$'s backward is the identity. The input $X$ feeds every rank's shard:
$\partial L/\partial X = \sum_r \frac{\partial L}{\partial U_r} W_{1,r}$ with $U_r$ rank $r$'s
pre-activation slice. Each rank computes one term, and $f$'s backward all-reduces them so every
rank gets the full gradient for the residual stream. Weight gradients need no communication:
$\partial L/\partial W_{1,r}$ and $\partial L/\partial W_{2,r}$ only involve rank $r$'s slices.

### Complexity

- Per rank and block: weight memory and matmul FLOPs divide by $W$; attention-score and softmax
  work divides by $W$ too, because heads are split.
- Communication per block: one all-reduce of $B \cdot T \cdot d$ elements in forward and one in
  backward; a ring all-reduce makes every rank send $2\frac{W-1}{W}BTd$ elements
  (see `v4/systems/ring-allreduce`). Unlike DDP's gradient all-reduce it sits on the critical
  path — the next layer needs its result — so hiding it takes extra machinery (splitting the
  matmuls and the collective into overlapping chunks).
- Activations: $H$ and the per-head tensors are sharded; the $(B, T, d)$ block inputs and outputs
  — and the LayerNorm/dropout between blocks — are replicated on every rank.

### Common bugs

- **Wrong dimension.** Megatron's $A$ is $W^\top$: "split $A$ by columns" means `weight.chunk(W, dim=0)`.
- **Communicating after the column-parallel layer.** Its outputs are disjoint feature slices;
  all-reducing them adds unrelated features (and costs a second collective).
- **Row-parallel bias on every rank** → the output is off by $(W-1)\,b$.
- **Averaging instead of summing** — confusing TP partial sums with DDP gradient averaging.
- **Splitting the fused QKV matrix into $W$ contiguous pieces**: rank 0 gets mostly query rows
  and the last rank mostly value rows, so a rank's q, k and v belong to different heads.
  Megatron avoids this by storing QKV grouped per head (per query group with GQA:
  `[ng, (np/ng + 2) * hn]`), so its plain column split is head-aligned; with the
  `[W_q; W_k; W_v]` layout you must split each block separately.
- **Reshaping a rank's q/k/v with the global head count**, or assigning heads to ranks in an order
  that does not match the columns of $W_o$ each rank holds.
- **Calling the collective outside an autograd Function**: with a non-differentiable all-reduce no
  gradient reaches the shards; with a differentiable sum every shard gets $W\times$ its gradient.
- **Forgetting $f$**: the forward is fine, but each rank's input gradient holds only its own
  shard's contribution.
- **Applying LayerNorm (or softmax over features) to a shard or a pre-reduction partial sum**:
  those need statistics over the full hidden vector.

### Interview follow-ups

1. **How much does TP communicate, and why keep it inside a node?** Per layer and micro-batch:
   two all-reduces of $b\,s\,h$ elements in forward and two in backward (Fig. 4). With a ring each
   costs every GPU $\approx 2\frac{W-1}{W}bsh$ elements sent — for $b = 1$, $s = 4096$, $h = 8192$,
   bf16, a 64 MiB tensor and about 112 MiB sent per GPU per all-reduce at $W = 8$. They block every
   layer, so TP needs NVLink/NVSwitch bandwidth (H100: 900 GB/s of NVLink per GPU counting both
   directions, 450 GB/s each way, versus about 50 GB/s each way for a 400 Gb/s InfiniBand NIC). Narayanan et al. (2021) conclude: use TP up to the number of
   GPUs in a server, pipeline parallelism across servers.
2. **What is sequence parallelism?** (Korthikanti et al., 2022.) Between the TP blocks, LayerNorm
   and dropout run on the full $(s, b, h)$ activation, replicated on every rank. Sequence
   parallelism shards those regions along the sequence and replaces this problem's $f$ and $g$ by
   a new conjugate pair (the paper calls them $g$ and $\bar g$): all-gather forward /
   reduce-scatter backward when entering a TP block, reduce-scatter forward / all-gather backward
   when leaving it. A ring all-reduce *is* a reduce-scatter plus an all-gather, so the bandwidth
   is unchanged while those activations shrink by $W$.
3. **How are the embedding and the LM head parallelised?** Vocab-parallel: each rank holds
   $V/W$ rows of the embedding; ids outside its range look up zeros, then an all-reduce ($g$)
   combines them. For the output layer, all-gathering the logits would move $b \times s \times v$
   elements, so Megatron fuses the column-parallel logits with the cross-entropy and only
   reduces $b \times s$-sized quantities (the max, the sum of exponentials and the target logit).
4. **Could you put a LayerNorm inside the TP region?** No: it needs the mean and variance of the
   full hidden vector, and a feature shard or a pre-reduction partial sum gives different
   statistics. GeLU is fine on a column shard only because it is elementwise. Norms therefore sit
   at block boundaries, where every token's hidden vector is complete (sequence parallelism runs
   them on sequence shards and all-gathers right after them).
5. **How does TP compose with DP, PP and FSDP?** Every GPU belongs to a TP group (inside a node),
   a pipeline group and a data-parallel group — "3D parallelism". Gradients of a TP shard are
   averaged only across the DP ranks that hold the *same* shard (all-reduce for DDP,
   reduce-scatter for ZeRO/FSDP). PyTorch expresses this as a 2-D `DeviceMesh` with dims
   `("dp", "tp")`: `parallelize_module` with `ColwiseParallel` / `RowwiseParallel` on the "tp"
   dim, FSDP2 `fully_shard` on the "dp" dim.
6. **What if there are fewer KV heads than TP ranks (GQA)?** Query heads still split evenly, but a
   KV head must live on every rank whose query heads read it, so KV heads are replicated (vLLM's
   `QKVParallelLinear` sets `num_kv_head_replicas = tp_size // num_kv_heads` when
   `tp_size >= num_kv_heads`). With `num_kv_heads >= W`, KV heads are split like query heads,
   keeping each query group on one rank.